# DataSalud Perú — análisis Big Data con PySpark

**Objetivo:** procesar los CSV limpios de La Libertad, medir tiempos reales en el entorno donde se ejecute el notebook y generar agregados reproducibles. Este notebook no inventa una comparación contra SQL Server: los tiempos de ambos motores deben medirse por separado en condiciones documentadas.

Antes de ejecutar: instala `pyspark` y coloca los archivos generados por `scripts/01_perfil_etl.py` en `resultados/etl/` o ajusta `BASE`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from time import perf_counter
import os, json, platform

BASE = os.path.abspath(os.path.join('..', 'resultados', 'etl'))
DENGUE_PATH = os.path.join(BASE, 'dengue_la_libertad_limpio.csv')
BIRTHS_PATH = os.path.join(BASE, 'nacimientos_la_libertad_limpio.csv')
assert os.path.exists(DENGUE_PATH), f'No existe: {DENGUE_PATH}'
assert os.path.exists(BIRTHS_PATH), f'No existe: {BIRTHS_PATH}'

spark = (SparkSession.builder
         .appName('DataSalud_Peru_LaLibertad')
         .config('spark.sql.shuffle.partitions', '8')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
print('Python:', platform.python_version())

### 1. Lectura y materialización de los datasets limpios

In [ ]:
t0 = perf_counter()
dengue = (spark.read.option('header', True).option('sep', ';').option('quote', '"')
          .option('encoding', 'UTF-8').csv(DENGUE_PATH))
nacimientos = (spark.read.option('header', True).option('sep', ';').option('quote', '"')
               .option('encoding', 'UTF-8').csv(BIRTHS_PATH))
# Cache y count fuerzan la lectura; el tiempo incluye parseo y materialización.
dengue.cache(); nacimientos.cache()
n_dengue = dengue.count(); n_nacimientos = nacimientos.count()
read_seconds = perf_counter() - t0
print({'filas_dengue': n_dengue, 'filas_nacimientos': n_nacimientos,
       'lectura_y_count_segundos': round(read_seconds, 4)})

### 2. KPI de dengue por provincia

In [ ]:
t0 = perf_counter()
dengue_kpi = (dengue.groupBy('provincia')
              .agg(F.count('*').alias('casos'),
                   F.avg(F.col('edad').cast('double')).alias('edad_promedio'))
              .orderBy(F.desc('casos')))
dengue_kpi.show(50, truncate=False)
dengue_seconds = perf_counter() - t0
print('Tiempo real de agregación dengue (s):', round(dengue_seconds, 4))

### 3. KPI de nacimientos por año y sexo

In [ ]:
t0 = perf_counter()
birth_kpi = (nacimientos.groupBy('FecNac_Año', 'sexo_nacido')
             .agg(F.count('*').alias('nacimientos'),
                  F.avg(F.col('PESO_NACIDO').cast('double')).alias('peso_promedio_g'),
                  F.avg(F.col('Edad_Madre').cast('double')).alias('edad_materna_promedio'))
             .orderBy('FecNac_Año', 'sexo_nacido'))
birth_kpi.show(100, truncate=False)
birth_seconds = perf_counter() - t0
print('Tiempo real de agregación nacimientos (s):', round(birth_seconds, 4))

### 4. Exportar métricas de esta ejecución

In [ ]:
metrics = {
    'spark_version': spark.version,
    'filas_dengue': int(n_dengue),
    'filas_nacimientos': int(n_nacimientos),
    'lectura_y_count_segundos': round(float(read_seconds), 4),
    'agregacion_dengue_segundos': round(float(dengue_seconds), 4),
    'agregacion_nacimientos_segundos': round(float(birth_seconds), 4),
    'nota': 'Tiempos medidos en el entorno donde se ejecutó este notebook; no generalizar a otros equipos.'
}
with open(os.path.join(BASE, 'spark_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print(metrics)
# Guardar agregados para inspección/entrega
dengue_kpi.coalesce(1).write.mode('overwrite').option('header', True).csv(os.path.join(BASE, 'spark_kpi_dengue'))
birth_kpi.coalesce(1).write.mode('overwrite').option('header', True).csv(os.path.join(BASE, 'spark_kpi_nacimientos'))
spark.stop()

## Cómo reportar el benchmark

Registra el equipo, CPU/RAM, versión de SQL Server/Spark, número de filas, tiempo de lectura/carga y tiempo de la misma agregación. Repite cada consulta tres veces e informa mediana; separa primera ejecución (arranque/caché fría) de ejecuciones posteriores. No compares un Spark local de un solo nodo con un clúster distribuido sin aclararlo.